In [4]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, dayofmonth, current_timestamp

###########################################################################
# Python Spark ETL job to transform raw data in json format to parquet
# For S3:
#   raw: s3://order-bucket/raw/orders/<timestamp>.json
#   etl: s3://order-bucket/processed/orders/year=2026/month=04/day=18/*.parquet
# For local file system:
#   raw: orders/file.json
#   etl: orders-processed/year=2026/month=04/day=18/*.parquet
###########################################################################

spark = SparkSession.builder.appName("OrderETL").getOrCreate()

# Read raw JSON from S3
#df = spark.read.json("s3://order-bucket/raw/orders/")
df = spark.read.json("orders/")  # this folder must exists and has at least one valid json file
df.printSchema()

# Basic transformation
df_clean = df \
    .withColumn("ingest_time", current_timestamp()) \
    .filter(col("orderId").isNotNull())

# Add partition columns
df_partitioned = df_clean \
    .withColumn("year", year("ingest_time")) \
    .withColumn("month", month("ingest_time")) \
    .withColumn("day", dayofmonth("ingest_time"))

# Write as Parquet
# It's import to include .par
df_partitioned.write \
    .mode("overwrite") \
    .partitionBy("year", "month", "day") \
    .parquet("orders-processed/")
    #.parquet("s3://order-bucket/processed/orders/")

print("DF.write.parquet: SUCCESSFUL")

SyntaxError: unexpected character after line continuation character (3146050836.py, line 36)